In [1]:
import re
import random
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
knowledge_base = [
    {
        "question": "hello",
        "answer": "Hello! Welcome. How can I help you?"
    },
    {
        "question": "what is natural language processing",
        "answer": "Natural Language Processing (NLP) enables computers to understand human language."
    },
    {
        "question": "what is machine learning",
        "answer": "Machine learning enables computers to learn patterns from data."
    },
    {
        "question": "what is deep learning",
        "answer": "Deep learning uses multi-layer neural networks to learn complex patterns."
    },
    {
        "question": "what is artificial intelligence",
        "answer": "Artificial Intelligence enables machines to perform tasks requiring human-like intelligence."
    },
    {
        "question": "what is python",
        "answer": "Python is a popular programming language used in web development, automation, data science, and AI."
    },
    {
        "question": "what is a chatbot",
        "answer": "A chatbot is a program that communicates with users through text or speech."
    },
    {
        "question": "what is tokenization",
        "answer": "Tokenization divides text into smaller units called tokens, such as words or sentences."
    },
    {
        "question": "what is sentiment analysis",
        "answer": "Sentiment analysis identifies opinions or emotions expressed in text."
    },
    {
        "question": "what is text classification",
        "answer": "Text classification assigns text to categories based on its content."
    },
    {
        "question": "what is tf idf",
        "answer": "TF-IDF measures how important a word is in a document relative to a collection of documents."
    },
    {
        "question": "what is word embedding",
        "answer": "Word embeddings represent words as numerical vectors that capture semantic relationships."
    },
    {
        "question": "what is text summarization",
        "answer": "Text summarization creates a shorter version of a document while preserving important information."
    },
    {
        "question": "thank you",
        "answer": "You're welcome! Feel free to ask another question."
    },
    {
        "question": "bye",
        "answer": "Goodbye! Have a great day."
    }
]

print("Total knowledge-base entries:", len(knowledge_base))

Total knowledge-base entries: 15


In [3]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

questions = [
    preprocess_text(item["question"])
    for item in knowledge_base
]

print("Preprocessed questions:")
for question in questions:
    print("-", question)

Preprocessed questions:
- hello
- what is natural language processing
- what is machine learning
- what is deep learning
- what is artificial intelligence
- what is python
- what is a chatbot
- what is tokenization
- what is sentiment analysis
- what is text classification
- what is tf idf
- what is word embedding
- what is text summarization
- thank you
- bye


In [4]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    lowercase=True
)

question_vectors = vectorizer.fit_transform(questions)

print("TF-IDF matrix shape:", question_vectors.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))

TF-IDF matrix shape: (15, 49)
Vocabulary size: 49


In [5]:
def chatbot_response(user_input, threshold=0.25):
    cleaned_input = preprocess_text(user_input)

    if not cleaned_input:
        return "Please enter a question."

    greetings = {"hi", "hey", "hello", "good morning", "good evening"}
    if cleaned_input in greetings:
        return "Hello! I'm your NLP chatbot. Ask me something about NLP or AI."

    user_vector = vectorizer.transform([cleaned_input])

    similarities = cosine_similarity(
        user_vector,
        question_vectors
    ).flatten()

    best_index = similarities.argmax()
    best_score = similarities[best_index]

    if best_score < threshold:
        return "Sorry, I don't know the answer yet. Please try another question."

    return knowledge_base[best_index]["answer"]

In [6]:
test_questions = [
    "Hello",
    "Explain natural language processing",
    "Tell me about machine learning",
    "Define sentiment analysis",
    "What is TF-IDF?",
    "Explain text summarization",
    "Tell me about quantum physics"
]

for question in test_questions:
    print("User:", question)
    print("Bot:", chatbot_response(question))
    print("-" * 60)

User: Hello
Bot: Hello! I'm your NLP chatbot. Ask me something about NLP or AI.
------------------------------------------------------------
User: Explain natural language processing
Bot: Natural Language Processing (NLP) enables computers to understand human language.
------------------------------------------------------------
User: Tell me about machine learning
Bot: Machine learning enables computers to learn patterns from data.
------------------------------------------------------------
User: Define sentiment analysis
Bot: Sentiment analysis identifies opinions or emotions expressed in text.
------------------------------------------------------------
User: What is TF-IDF?
Bot: TF-IDF measures how important a word is in a document relative to a collection of documents.
------------------------------------------------------------
User: Explain text summarization
Bot: Text summarization creates a shorter version of a document while preserving important information.
----------------

In [7]:
print("NLP Chatbot started!")
print("Ask me about NLP, AI, Python, or machine learning.")
print("Type 'exit' to stop.\n")

while True:
    user_input = input("You: ").strip()

    if user_input.lower() in {"exit", "quit", "stop"}:
        print("Bot: Goodbye! Have a great day.")
        break

    response = chatbot_response(user_input)
    print("Bot:", response, "\n")

NLP Chatbot started!
Ask me about NLP, AI, Python, or machine learning.
Type 'exit' to stop.



You:  hlo


Bot: Sorry, I don't know the answer yet. Please try another question. 



You:  what is nlp


Bot: Python is a popular programming language used in web development, automation, data science, and AI. 



You:  what is machine learning


Bot: Machine learning enables computers to learn patterns from data. 



You:  exit


Bot: Goodbye! Have a great day.


In [8]:
def inspect_similarity(user_input):
    user_vector = vectorizer.transform([
        preprocess_text(user_input)
    ])

    scores = cosine_similarity(
        user_vector,
        question_vectors
    ).flatten()

    results = pd.DataFrame({
        "Knowledge Base Question": [
            item["question"] for item in knowledge_base
        ],
        "Similarity Score": scores
    })

    return results.sort_values(
        by="Similarity Score",
        ascending=False
    ).reset_index(drop=True)

display(inspect_similarity("How does a computer learn from data?").head(5))

,Knowledge Base Question,Similarity Score
0,hello,0.0
1,what is natural language processing,0.0
2,what is machine learning,0.0
3,what is deep learning,0.0
4,what is artificial intelligence,0.0


In [9]:
def add_knowledge(question, answer):
    global knowledge_base, questions, vectorizer, question_vectors

    question = question.strip()
    answer = answer.strip()

    if not question or not answer:
        print("Question and answer cannot be empty.")
        return

    knowledge_base.append({
        "question": question,
        "answer": answer
    })

    questions = [
        preprocess_text(item["question"])
        for item in knowledge_base
    ]

    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        lowercase=True
    )

    question_vectors = vectorizer.fit_transform(questions)

    print("Knowledge base updated successfully!")

add_knowledge(
    "what is information retrieval",
    "Information retrieval finds relevant information from a collection of documents."
)

print(chatbot_response("Explain information retrieval"))

Knowledge base updated successfully!
Information retrieval finds relevant information from a collection of documents.


In [10]:
knowledge_df = pd.DataFrame(knowledge_base)

knowledge_df.to_csv(
    "chatbot_knowledge_base.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Knowledge base saved to chatbot_knowledge_base.csv")
display(knowledge_df.head())

Knowledge base saved to chatbot_knowledge_base.csv


,question,answer
0,hello,Hello! Welcome. How can I help you?
1,what is natural language processing,Natural Language Processing (NLP) enables comp...
2,what is machine learning,Machine learning enables computers to learn pa...
3,what is deep learning,Deep learning uses multi-layer neural networks...
4,what is artificial intelligence,Artificial Intelligence enables machines to pe...
